# News Wire Services - Step-by-Step Tutorial

This notebook walks through how to programmatically pull press releases from major **news wire services** using RSS feeds.

## Covered Services

1. **Business Wire** (owned by Berkshire Hathaway)
2. **PR Newswire** (Cision)
3. **GlobeNewswire** (Intrado)

## Why RSS Feeds?

All three services **block direct web scraping** (HTTP 403 errors), but they provide **official RSS feeds** for accessing press releases. These are:
- ✅ **Legitimate** — officially supported
- ✅ **Free** — no API key required
- ✅ **Reliable** — standardized XML format
- ✅ **Real-time** — updated as releases are published

## Focus: Biotech & Pharma

This tutorial focuses on fetching **biotechnology** and **pharmaceutical** press releases, including:
- Clinical trial announcements
- FDA approvals
- Drug development updates
- Regulatory milestones
- Funding and M&A news

---
## Step 1: Install Dependencies

We need `requests` for HTTP and `feedparser` for parsing RSS/Atom feeds.

In [ ]:
!pip install requests feedparser -q

In [ ]:
import requests
import feedparser
import json
from datetime import datetime
from collections import Counter

print("Dependencies loaded successfully!")

---
## Step 2: Understanding RSS Feed URLs

Each wire service provides RSS feeds filtered by industry or subject.

### Business Wire RSS Feeds

| Industry | RSS Feed URL |
|----------|-------------|
| Biotechnology | `https://www.businesswire.com/portal/site/home/template.PAGE/news/rss/?ndmConfigId=1001106` |
| Pharmaceutical | `https://www.businesswire.com/portal/site/home/template.PAGE/news/rss/?ndmConfigId=1001107` |
| Health | `https://www.businesswire.com/portal/site/home/template.PAGE/news/rss/?ndmConfigId=1001014` |
| FDA | `https://www.businesswire.com/portal/site/home/template.PAGE/news/rss/?ndmConfigId=1001348` |

### PR Newswire RSS Feeds

| Industry | RSS Feed URL |
|----------|-------------|
| Biotechnology | `https://www.prnewswire.com/rss/health-latest-news/biotechnology-list.rss` |
| Pharmaceutical | `https://www.prnewswire.com/rss/health-latest-news/pharmaceuticals-list.rss` |
| Medical Devices | `https://www.prnewswire.com/rss/health-latest-news/medical-devices-list.rss` |
| Health Care | `https://www.prnewswire.com/rss/health-latest-news/health-care-latest-news-list.rss` |

### GlobeNewswire RSS Feeds

| Keyword | RSS Feed URL |
|---------|-------------|
| Healthcare | `https://www.globenewswire.com/RssFeed/industry/Healthcare` |
| Biotech | `https://www.globenewswire.com/RssFeed/keyword/biotech` |
| Clinical Trial | `https://www.globenewswire.com/RssFeed/keyword/clinical%20trial` |

**Note:** PR Newswire is generally the most accessible and reliable.

---
## Step 3: Fetch and Parse an RSS Feed

Let's start with PR Newswire's biotechnology feed (the most accessible).

In [ ]:
# PR Newswire Biotechnology RSS feed
feed_url = "https://www.prnewswire.com/rss/health-latest-news/biotechnology-list.rss"

print(f"Fetching: {feed_url}")

# Fetch the feed
response = requests.get(feed_url, timeout=30, headers={
    "User-Agent": "Mozilla/5.0 (compatible; RSSReader/1.0)"
})

print(f"Status: {response.status_code}")
print(f"Content-Type: {response.headers.get('Content-Type', 'N/A')}")
print(f"Response size: {len(response.content)} bytes")

In [ ]:
# Parse the RSS feed with feedparser
feed = feedparser.parse(response.content)

print(f"Feed title: {feed.feed.get('title', 'N/A')}")
print(f"Feed description: {feed.feed.get('description', 'N/A')}")
print(f"Total entries: {len(feed.entries)}")
print(f"Parsing errors: {feed.bozo}")

---
## Step 4: Understand the RSS Entry Structure

RSS feeds contain entries with standardized fields:

| Field | Description |
|-------|-------------|
| `title` | Press release headline |
| `link` | URL to full press release |
| `published` | Publication date/time |
| `summary` | Brief description or excerpt |
| `id` or `guid` | Unique identifier |
| `tags` | Category tags (optional) |

Let's examine the first entry:

In [ ]:
# Print the first entry in detail
if feed.entries:
    first = feed.entries[0]
    
    print("First press release:")
    print("=" * 70)
    print(f"Title:     {first.get('title', 'N/A')}")
    print(f"Link:      {first.get('link', 'N/A')}")
    print(f"Published: {first.get('published', 'N/A')}")
    print(f"Summary:   {first.get('summary', 'N/A')[:200]}...")
    print(f"ID:        {first.get('id', first.get('guid', 'N/A'))}")
    
    # Show all available fields
    print(f"\nAvailable fields: {list(first.keys())}")

---
## Step 5: Parse Multiple Entries

Let's extract key information from all entries in the feed.

In [ ]:
def parse_rss_entries(feed):
    """
    Parse RSS feed entries into a clean list of dicts.
    """
    releases = []
    
    for entry in feed.entries:
        # Extract company name from title (before dash or colon)
        title = entry.get('title', '')
        company = ""
        
        if ' - ' in title:
            company = title.split(' - ')[0].strip()
        elif ': ' in title:
            parts = title.split(': ', 1)
            if len(parts[0]) < 50:  # Likely a company name
                company = parts[0].strip()
        
        release = {
            "title": title,
            "company": company,
            "link": entry.get('link', ''),
            "published": entry.get('published', ''),
            "summary": entry.get('summary', entry.get('description', '')),
            "guid": entry.get('id', entry.get('guid', '')),
        }
        releases.append(release)
    
    return releases


# Parse entries
releases = parse_rss_entries(feed)

print(f"Parsed {len(releases)} press releases")
print("\nFirst 5 releases:")
print("=" * 70)

for i, r in enumerate(releases[:5], 1):
    print(f"\n{i}. [{r['company'] or 'N/A'}]")
    print(f"   {r['title']}")
    print(f"   Published: {r['published']}")
    print(f"   Link: {r['link']}")

---
## Step 6: Fetch from Multiple Wire Services

Let's fetch from PR Newswire, GlobeNewswire, and Business Wire.

In [ ]:
# Define all feeds
feeds_to_fetch = {
    "PR Newswire - Biotech": "https://www.prnewswire.com/rss/health-latest-news/biotechnology-list.rss",
    "PR Newswire - Pharma": "https://www.prnewswire.com/rss/health-latest-news/pharmaceuticals-list.rss",
    "GlobeNewswire - Healthcare": "https://www.globenewswire.com/RssFeed/industry/Healthcare/feedTitle/GlobeNewswire%20-%20Healthcare",
    # Business Wire feeds may be blocked - included for reference
    # "Business Wire - Biotech": "https://www.businesswire.com/portal/site/home/template.PAGE/news/rss/?ndmConfigId=1001106",
}

all_releases = []

for name, url in feeds_to_fetch.items():
    print(f"\nFetching {name}...", end=" ")
    
    try:
        resp = requests.get(url, timeout=30, headers={
            "User-Agent": "Mozilla/5.0 (compatible; RSSReader/1.0)"
        })
        resp.raise_for_status()
        
        feed = feedparser.parse(resp.content)
        releases = parse_rss_entries(feed)
        
        # Add source tag
        for r in releases:
            r['source'] = name
        
        all_releases.extend(releases)
        print(f"got {len(releases)} releases")
        
    except Exception as e:
        print(f"FAILED: {e}")

print(f"\nTotal releases fetched: {len(all_releases)}")

---
## Step 7: Explore the Data

Let's look at the distribution by source and find the most active companies.

In [ ]:
# Count by source
source_counts = Counter(r['source'] for r in all_releases)

print("Press Releases by Wire Service:")
print("=" * 50)
for source, count in source_counts.most_common():
    print(f"  {source}: {count}")

In [ ]:
# Count by company (top 15)
company_counts = Counter(
    r['company'] for r in all_releases if r['company']
)

print("\nTop 15 Most Active Companies:")
print("=" * 50)
for company, count in company_counts.most_common(15):
    print(f"  {company}: {count}")

---
## Step 8: Filter by Keywords

Filter press releases for specific topics like FDA approvals, clinical trials, or drug development.

In [ ]:
# Keywords for FDA approvals and clinical trials
keywords = [
    "fda approval", "fda clearance", "fda granted",
    "clinical trial", "phase 1", "phase 2", "phase 3",
    "pivotal trial", "trial results", "data readout",
    "breakthrough therapy", "orphan drug", "fast track",
    "new drug application", "nda", "bla",
]

def filter_by_keywords(releases, keywords):
    """
    Filter releases by keywords in title or summary.
    """
    filtered = []
    for r in releases:
        text = (r['title'] + " " + r['summary']).lower()
        if any(kw.lower() in text for kw in keywords):
            filtered.append(r)
    return filtered


# Filter for clinical/FDA news
clinical_fda = filter_by_keywords(all_releases, keywords)

print(f"Clinical & FDA-related releases: {len(clinical_fda)}")
print("\nExamples:")
print("=" * 70)

for r in clinical_fda[:10]:
    print(f"\n[{r['source']}] {r['company']}")
    print(f"  {r['title']}")
    print(f"  {r['published']}")

---
## Step 9: Filter by Company

Track press releases from specific companies you're interested in.

In [ ]:
# Notable biotech/pharma companies to track
target_companies = [
    "BeiGene", "Moderna", "BioNTech", "Regeneron",
    "Vertex", "Biogen", "Gilead", "Amgen",
    "Sanofi", "Novartis", "Roche", "AstraZeneca",
]

def filter_by_company(releases, companies):
    """
    Filter releases by company name (partial match).
    """
    filtered = []
    for r in releases:
        if any(comp.lower() in r['company'].lower() for comp in companies):
            filtered.append(r)
    return filtered


# Filter by target companies
company_releases = filter_by_company(all_releases, target_companies)

print(f"Releases from target companies: {len(company_releases)}")
print("\nExamples:")
print("=" * 70)

for r in company_releases[:10]:
    print(f"\n[{r['company']}]")
    print(f"  {r['title']}")
    print(f"  {r['published']}")
    print(f"  {r['link']}")

---
## Step 10: Save Results to JSON

Export the filtered results for further analysis or ingestion into a data pipeline.

In [ ]:
# Save all releases
output = {
    "fetched_at": datetime.utcnow().isoformat(),
    "source": "Wire Services (PR Newswire, GlobeNewswire)",
    "total_releases": len(all_releases),
    "releases": all_releases,
}

with open("wire_all_releases.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Saved {len(all_releases)} releases to wire_all_releases.json")

In [ ]:
# Save clinical/FDA-filtered releases
clinical_output = {
    "fetched_at": datetime.utcnow().isoformat(),
    "source": "Wire Services",
    "filter": "Clinical trials & FDA approvals",
    "total_releases": len(clinical_fda),
    "releases": clinical_fda,
}

with open("wire_clinical_fda.json", "w", encoding="utf-8") as f:
    json.dump(clinical_output, f, indent=2, ensure_ascii=False)

print(f"Saved {len(clinical_fda)} clinical/FDA releases to wire_clinical_fda.json")

---
## Step 11: Using the Parser Module

For production use, import the `WireServiceParser` class which wraps everything into a clean interface.

In [ ]:
import sys
sys.path.insert(0, ".")

from wire_parser import WireServiceParser

# Initialize
parser = WireServiceParser()

# Fetch from PR Newswire (most reliable)
releases = parser.fetch_pr_newswire()
print(f"\nFetched {len(releases)} releases from PR Newswire")

# Filter for FDA-related news
fda_news = parser.filter_by_keywords(releases, ["fda", "approval", "clearance"])
print(f"FDA-related: {len(fda_news)}")

# Filter for specific companies
company_news = parser.filter_by_company(releases, ["Moderna", "BioNTech", "Pfizer"])
print(f"Target companies: {len(company_news)}")

# Show examples
for r in fda_news[:3]:
    print(f"\n  [{r.source}] {r.company}")
    print(f"  {r.title}")
    print(f"  {r.published_at}")
    print(f"  {r.link}")

---
## Summary

### What We Learned

1. **Wire services block web scraping** but offer official **RSS feeds**
2. **PR Newswire is the most accessible** — use it as your primary source
3. **GlobeNewswire is a good backup** with keyword-based feeds
4. **Business Wire feeds may be restricted** — requires proper headers or access
5. **RSS feeds are updated in real-time** as press releases are published
6. **No API key required** — completely free to use

### RSS Feed URLs (Quick Reference)

**PR Newswire (Recommended):**
```
Biotechnology:  https://www.prnewswire.com/rss/health-latest-news/biotechnology-list.rss
Pharmaceutical: https://www.prnewswire.com/rss/health-latest-news/pharmaceuticals-list.rss
Medical Devices: https://www.prnewswire.com/rss/health-latest-news/medical-devices-list.rss
```

**GlobeNewswire:**
```
Healthcare: https://www.globenewswire.com/RssFeed/industry/Healthcare
Biotech:    https://www.globenewswire.com/RssFeed/keyword/biotech
```

**Business Wire:**
```
Biotechnology:  https://www.businesswire.com/portal/site/home/template.PAGE/news/rss/?ndmConfigId=1001106
Pharmaceutical: https://www.businesswire.com/portal/site/home/template.PAGE/news/rss/?ndmConfigId=1001107
```

### Standard RSS Entry Fields

| Field | Description |
|-------|-------------|
| `title` | Press release headline |
| `link` | URL to full release |
| `published` | Publication timestamp |
| `summary` | Brief excerpt |
| `id` or `guid` | Unique identifier |

### Best Practices

1. **Use PR Newswire as primary source** — most reliable and accessible
2. **Add delays between requests** — be respectful of rate limits (1-2 seconds)
3. **Parse with feedparser** — handles XML quirks and malformed feeds
4. **Filter by keywords** — FDA, clinical trial, phase 1/2/3, approval, etc.
5. **Track by company** — monitor specific biotech/pharma firms
6. **Check for duplicates** — same release may appear in multiple feeds

### Limitations

- **No historical data** — RSS feeds typically show last 20-100 releases
- **No full-text** — summaries only; must follow link for complete release
- **Rate limiting** — excessive requests may result in temporary blocks
- **Feed delays** — releases appear in RSS within minutes of publication
- **Business Wire may require authorization** — some feeds are restricted